# ASG-WM / FaithCast — **Eval All (ours)** -> paper assets

Evaluates **our** model from trained checkpoints -> **`paper_assets.zip`** (every value `*_results.json`/`tables/*.tex` + image `figures/*.pdf`/`*.png`).

**Pipeline:** settings -> install -> access preflight (SEVIR+NEXRAD+MRMS) -> skill -> faithfulness -> figures+gallery -> zip. OOD (NEXRAD/MRMS) uses the 120-min headline window.

Run `train.ipynb` first (or unzip `train_results.zip` into `artifacts/`). Baselines stay `[TBR]` until trained — this evaluates ours only.

In [ ]:
# --- 1. Locate / clone the repo ----------------------------------------------
# On Colab: set ASGWM_REPO_URL to your git remote and this cell clones it.
# Locally: run this notebook from the repo root (the cell is a no-op).
import os, sys, subprocess, pathlib
REPO_URL = os.environ.get("ASGWM_REPO_URL", "")
REPO_DIR = "Forecasting-Through-Meteorological-Reasoning"
if not pathlib.Path("src/asgwm").exists():
    if not pathlib.Path(REPO_DIR, "src", "asgwm").exists():
        assert REPO_URL, "Set ASGWM_REPO_URL (Colab) or launch from the repo root."
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
assert pathlib.Path("src/asgwm").exists(), f"Run from repo root; cwd={os.getcwd()}"
print("repo root:", os.getcwd())


In [ ]:
# --- 2. Settings -------------------------------------------------------------
import subprocess, sys, itertools, os
# In-distribution = SEVIR test; out-of-distribution generalization = NEXRAD, MRMS.
DATASETS = ["synthetic"]     # e.g. ["sevir", "nexrad", "mrms"] for the full paper run
N_EVAL   = 32                # eval events per dataset
CFG      = "src/configs/default.yaml"
OOD = {"nexrad", "mrms"}     # use the robust 120-min headline window for OOD downloads
print("datasets:", DATASETS)


In [ ]:
# --- 3. Install dependencies (core + backends for every dataset evaluated) ----
def _pip(*pkgs):
    if pkgs:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
EXTRAS = {
    "synthetic": [],
    "sevir":  ["s3fs", "h5py"],                       # SEVIR HDF5 on s3://sevir
    "nexrad": ["arm-pyart", "boto3"],                 # NEXRAD L2 polar read + gridding
    "mrms":   ["boto3", "xarray", "cfgrib", "eccodes"],  # MRMS gzipped GRIB2
}
_pip("-e", "./src")
_pip("-r", "src/requirements.txt")
need = sorted({p for d in DATASETS for p in EXTRAS[d]})
_pip(*need)
print("installed core + extras:", need or "(none)")


In [ ]:
# --- 4. Data-access preflight for ALL real datasets (HTTPS; no heavy deps) ----
# Always reports SEVIR/NEXRAD/MRMS download-readiness (OOD checked at the 120-min window).
subprocess.run([sys.executable, "datasets/check_access.py",
                "--datasets", "sevir", "nexrad", "mrms",
                "--override", "data.out_frames=12"], check=False)


In [ ]:
# --- 5. Skill + faithfulness + figures, per dataset --------------------------
# Each dataset writes to its own results dir so OOD never clobbers the main tables.
# OOD downloads/evals use the 120-min headline horizon (out_frames=12) for robust coverage.
def ov(**kw):
    return list(itertools.chain.from_iterable(["--override", f"{k}={v}"] for k, v in kw.items()))
def run(script, results_dir, dataset, extra=None):
    common = ov(**{"data.dataset": dataset, "eval.n_eval_events": N_EVAL, "paths.results": results_dir})
    if dataset in OOD:
        common += ov(**{"data.out_frames": 12})
    cmd = [sys.executable, f"src/scripts/{script}", "--config", CFG] + common + (extra or [])
    print(">>", " ".join(cmd)); sys.stdout.flush(); subprocess.run(cmd, check=True)

RESULT_DIRS = {}
for ds in DATASETS:
    rdir = f"artifacts/results_{ds}"; RESULT_DIRS[ds] = rdir
    print(f"\n===== evaluating ASG-WM on: {ds}  ->  {rdir} =====")
    run("40_eval_skill.py",        rdir, ds)
    run("41_eval_faithfulness.py", rdir, ds)
    run("42_make_figures.py",      rdir, ds, ["--gallery"])


In [ ]:
# --- 6. Bundle paper assets (values + images) -> paper_assets.zip -------------
import glob, os, zipfile
OUT = "paper_assets.zip"
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for ds, rdir in RESULT_DIRS.items():
        for p in glob.glob(os.path.join(rdir, "**", "*"), recursive=True):
            if os.path.isfile(p) and p.lower().endswith((".json", ".tex", ".pdf", ".png", ".csv")):
                z.write(p)
print(f"wrote {OUT}  ({os.path.getsize(OUT)/1e6:.1f} MB)" if os.path.exists(OUT) else "nothing to zip")
print("contains *_results.json + tables/*.tex + figures/*.pdf|png per dataset")


**Done.** Download `paper_assets.zip` — drop `figures/*.pdf` + `tables/*.tex` into `paper/`.